In [1]:
from openai import OpenAI
# Basta puntare al server vLLM locale invece che alle API pubbliche
client = OpenAI(
    base_url="http://localhost:8000/v1",  # <--- UNICA MODIFICA
    api_key="EMPTY",
)

response = client.chat.completions.create(
    model="unsloth/SmolLM3-3B-bnb-4bit",
    messages=[{"role": "user", "content": "Ciao sono Andrea!"}],
    extra_body={"chat_template_kwargs":{"enable_thinking": False}},
)

# Stampa la risposta e i token usati
print(response.choices[0].message.content)
print("\n--- Dettagli Tecnici ---")
print(f"Token totali: {response.usage.total_tokens}")
print(f"Modello usato: {response.model}")

Ciao Andrea! Piacere di conoscerti. Come posso aiutarti oggi?

--- Dettagli Tecnici ---
Token totali: 95
Modello usato: unsloth/SmolLM3-3B-bnb-4bit


In [2]:
#NB NB NB Se devo fare interrogazioni successive devo inserire nella lista di messaggi la risposta dell'assistente e la successiva domanda dell'utente
response = client.chat.completions.create(
    model="unsloth/SmolLM3-3B-bnb-4bit",
    messages=[{"role": "user", "content": "Ciao sono Andrea!"}, {"role":"assistant", "content": response.choices[0].message.content}, {"role": "user", "content": "Qual'è il mio nome?"}],
    extra_body={"chat_template_kwargs":{"enable_thinking": False}},
)
print(response.choices[0].message.content)
print("\n--- Dettagli Tecnici ---")
print(f"Token totali: {response.usage.total_tokens}")
print(f"Modello usato: {response.model}")

Il tuo nome è Andrea! Ho notato che hai detto "Ciao sono Andrea!" all'inizio della nostra conversazione.

--- Dettagli Tecnici ---
Token totali: 143
Modello usato: unsloth/SmolLM3-3B-bnb-4bit


In [3]:
#Se non aggiungo il risultato precedente alla lista di messaggi il sistema non sa nulla, non mantiene in nessun modo la storia dei messaggi
response = client.chat.completions.create(
    model="unsloth/SmolLM3-3B-bnb-4bit",
    messages=[{"role": "user", "content": "Qual'è il mio nome'"}],
    extra_body={"chat_template_kwargs":{"enable_thinking": True}},
)

# Stampa la risposta e i token usati
print(response.choices[0].message.content)
print("\n--- Dettagli Tecnici ---")
print(f"Token totali: {response.usage.total_tokens}")
print(f"Modello usato: {response.model}")

<think>
Okay, the user wrote "Qual'è il mio nome?" which means "What is my name?" in Italian. I need to figure out how to respond appropriately.

First, I should acknowledge the question. Since the user is asking for their own name, they might be referring to a specific person or entity. But without additional context, it's hard to know. Maybe they're testing the system, or perhaps they want to play a game where they ask for their name.

I should consider possible scenarios. If this is a chatbot interaction, the user might be trying to verify if the system can respond correctly. Alternatively, it could be part of a role-playing game or a scenario where the system is supposed to remember or generate a name.

Another angle is that the user might be asking in a hypothetical situation, like in a story or a game where they need to create a character. In that case, the response should be flexible, allowing the user to provide their name or create one.

I need to make sure my response is help

# Modalità streaming

In [5]:
for chunk in client.chat.completions.create(
    model="unsloth/SmolLM3-3B-bnb-4bit",
    messages=[{"role": "user", "content": "Ciao vLLM!"}],
    extra_body={"chat_template_kwargs":{"enable_thinking": True}},
    stream=True
):
    print(chunk.choices[0].delta.content, end='')
# Stampa la risposta e i token usati


<think>
Okay, the user just said "Ciao vLLM!" which is Italian for "Hello vLLM!". I should respond in Italian since the greeting is in Italian. Let me make sure to greet them back in the same language. I'll use a friendly and welcoming tone.

Also, I need to check if there's any context I should be aware of. The user might be testing my language proficiency or just starting a conversation. Since they didn't provide any specific questions or topics, I should ask how I can assist them. Keeping the response open-ended will encourage them to elaborate on what they need help with.

I should avoid making assumptions and stick to providing a helpful and welcoming response. Let me put that together in Italian.
</think>

Ciao! Come posso aiutarti oggi? Hai qualche domanda specifica o un argomento di cui vorresti parlare?

## Metriche

2. Le Metriche Chiave (Cosa monitorare)
Le metriche più importanti che vLLM ti restituisce sono:

vllm:gpu_cache_usage_perc: Questa è la metrica "regina". Ti dice quanto è piena la memoria della GPU gestita da PagedAttention.

Perché è utile: Se è al 95%, stai usando la GPU al massimo (ottimo). Se è bassa, puoi aumentare il batch size.

vllm:num_requests_running: Quante richieste sta elaborando in parallelo in questo istante.

vllm:num_requests_waiting: Quante richieste sono in coda (perché la GPU è piena).

Perché è utile: Se questo numero sale, ti serve un'altra GPU.

vllm:time_to_first_token_seconds: Quanto tempo passa da quando l'utente preme invio a quando vede la prima parola (latenza percepita).

vllm:generation_tokens_total: La velocità pura (throughput).

In [29]:
import requests
requests.get('http://localhost:8000/metrics').text

'# HELP python_gc_objects_collected_total Objects collected during gc\n# TYPE python_gc_objects_collected_total counter\npython_gc_objects_collected_total{generation="0"} 23335.0\npython_gc_objects_collected_total{generation="1"} 3837.0\npython_gc_objects_collected_total{generation="2"} 1389.0\n# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC\n# TYPE python_gc_objects_uncollectable_total counter\npython_gc_objects_uncollectable_total{generation="0"} 0.0\npython_gc_objects_uncollectable_total{generation="1"} 0.0\npython_gc_objects_uncollectable_total{generation="2"} 0.0\n# HELP python_gc_collections_total Number of times this generation was collected\n# TYPE python_gc_collections_total counter\npython_gc_collections_total{generation="0"} 1608.0\npython_gc_collections_total{generation="1"} 146.0\npython_gc_collections_total{generation="2"} 10.0\n# HELP python_info Python platform information\n# TYPE python_info gauge\npython_info{implementation="CPython"